# Example 0.13.3 / 4: Exercise 18-2: A Fence and a Bad Starting Point

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 18-2: A Fence and a Bad Starting Point". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 18-2: A Fence and a Bad Starting Point

Exercise 18-2 maximizes the area $XY$ of a rectangular field enclosed by 40 meters of fence. The
book reports that a local solver started at $X = Y = 0$ returned an area of 0, and asks why. At
$(0,0)$ the gradient of $XY$, which is $(Y, X)$, is zero, so a method that looks for a point where
the gradient conditions hold can stop there. The objective $XY$ is neither concave nor convex, so
this is a nonconvex problem (Pitfalls of Nonlinear Programming (AMPL lab documentation, section 11.4)).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param fence > 0;                                   # meters of fence
var Xfield >= 0;                                   # width, meters
var Yfield >= 0;                                   # length, meters
maximize Area: Xfield * Yfield;                    # square meters
subject to Enclose: 2*Xfield + 2*Yfield <= fence;  # perimeter within the fence

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["fence"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
print("HiGHS:", ampl.solve_result, ampl.var["Xfield"].value(), ampl.var["Yfield"].value())
ampl.option["ipopt_options"] = "print_level=0"
ampl.solve(solver="ipopt", verbose=False)
print("Ipopt:", ampl.solve_result, round(ampl.var["Xfield"].value(), 6), round(ampl.var["Yfield"].value(), 6))

HiGHS 1.14.0: optimal solution; objective 0
0 simplex iterations
0 barrier iterations
HiGHS: solved 0.0 0.0
Ipopt: solved 10.0 10.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param fence := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Xfield, Yfield;
option solver ipopt;
option ipopt_options 'print_level=0';
solve;
display Xfield, Yfield, Area;

HiGHS 1.14.0: optimal solution; objective 0
0 simplex iterations
0 barrier iterations
Xfield = 0
Yfield = 0

Ipopt 3.12.13: print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
Xfield = 10
Yfield = 10
Area = 100

